# Q3: CycleGAN — Unpaired Sketch ↔ Photo Translation
**Course:** Generative AI (AI4009) | **Assignment:** 03 | **Platform:** Kaggle T4 x2

## Cell 1 — Imports

In [1]:
import os, pickle, random, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

Device: cuda
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## Cell 2 — Hyperparameters

In [2]:
IMG_SIZE      = 128
BATCH_SIZE    = 4       # CycleGAN is memory-heavy
LR            = 0.0002
BETAS         = (0.5, 0.999)
EPOCHS        = 30
LAMBDA_CYCLE  = 10     # cycle consistency weight
LAMBDA_ID     = 5      # identity loss weight
N_RES_BLOCKS  = 6      # ResNet blocks (6 instead of 9 for Kaggle)
SAVE_EVERY    = 5

os.makedirs('outputs/cyclegan/AtoB', exist_ok=True)
os.makedirs('outputs/cyclegan/BtoA', exist_ok=True)
os.makedirs('checkpoints',           exist_ok=True)
print('Config ready.')

Config ready.


## Cell 3 — Find Unpaired Datasets

In [3]:
# Sketchy Dataset — hardcoded paths
# Structure inside data/: photo-train, sketch-triplet-train, photo-test, sketch-triplet-test
BASE         = '/kaggle/input/datasets/sharanyasundar/sketchy-dataset'
SKETCH_DIR   = os.path.join(BASE, 'data', 'sketch-triplet-train')
PHOTO_DIR    = os.path.join(BASE, 'data', 'photo-train')
VALID_EXTS   = {'.png', '.jpg', '.jpeg', '.webp'}

def collect_images(folder):
    """Recursively collect all image paths from a folder."""
    paths = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if os.path.splitext(f)[1].lower() in VALID_EXTS:
                paths.append(os.path.join(root, f))
    return paths

domain_A_paths = collect_images(SKETCH_DIR)   # Domain A = Sketches
domain_B_paths = collect_images(PHOTO_DIR)    # Domain B = Photos

if not domain_A_paths:
    raise FileNotFoundError(f'No sketch images found at: {SKETCH_DIR}')
if not domain_B_paths:
    raise FileNotFoundError(f'No photo images found at: {PHOTO_DIR}')

print(f'Domain A (Sketch): {len(domain_A_paths)} images')
print(f'Domain B (Photo) : {len(domain_B_paths)} images')
print('Paths verified!')

FileNotFoundError: No sketch images found at: /kaggle/input/datasets/sharanyasundar/sketchy-dataset/data/sketch-triplet-train

## Cell 4 — Unpaired Dataset

In [ ]:
class UnpairedDataset(Dataset):
    """Returns random (A, B) pairs — unpaired by design for CycleGAN."""
    def __init__(self, paths_A, paths_B, img_size=128):
        self.paths_A = paths_A
        self.paths_B = paths_B
        self.t = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self): return max(len(self.paths_A), len(self.paths_B))

    def _load(self, path):
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            return Image.new('RGB', (IMG_SIZE, IMG_SIZE), 128)

    def __getitem__(self, idx):
        img_A = self.t(self._load(self.paths_A[idx % len(self.paths_A)]))
        img_B = self.t(self._load(self.paths_B[random.randint(0, len(self.paths_B)-1)]))
        return img_A, img_B


MAX_PER_DOMAIN = 5000
random.shuffle(domain_A_paths); random.shuffle(domain_B_paths)
A_paths = domain_A_paths[:MAX_PER_DOMAIN]
B_paths = domain_B_paths[:MAX_PER_DOMAIN]

dataset    = UnpairedDataset(A_paths, B_paths, img_size=IMG_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=2,
                        pin_memory=True, drop_last=True)
print(f'Dataset length: {len(dataset)} | Batches/epoch: {len(dataloader)}')

## Cell 5 — Visualise Domains

In [ ]:
A_batch, B_batch = next(iter(dataloader))
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle('Domain A: Sketch (top) | Domain B: Photo (bottom)', fontsize=13)
for i in range(8):
    for row, imgs in enumerate([A_batch, B_batch]):
        img = imgs[i % imgs.size(0)].permute(1,2,0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[row,i].imshow(img); axes[row,i].axis('off')
plt.tight_layout()
plt.savefig('outputs/cyclegan/domain_samples.png', dpi=100)
plt.show()

## Cell 6 — ResNet Generator

In [ ]:
class ResNetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, 1, 0, bias=False),
            nn.InstanceNorm2d(dim), nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, 1, 0, bias=False),
            nn.InstanceNorm2d(dim)
        )
    def forward(self, x): return x + self.block(x)   # residual


class ResNetGenerator(nn.Module):
    """
    ResNet-based generator for CycleGAN.
    Input/Output: (B, 3, 128, 128)
    """
    def __init__(self, in_ch=3, out_ch=3, f=64, n_blocks=N_RES_BLOCKS):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, f, 7, 1, 0, bias=False),
            nn.InstanceNorm2d(f), nn.ReLU(True),
            # Downsample
            nn.Conv2d(f,   f*2, 3, 2, 1, bias=False), nn.InstanceNorm2d(f*2), nn.ReLU(True),
            nn.Conv2d(f*2, f*4, 3, 2, 1, bias=False), nn.InstanceNorm2d(f*4), nn.ReLU(True),
        ]
        # ResNet blocks
        for _ in range(n_blocks):
            layers.append(ResNetBlock(f*4))
        # Upsample
        layers += [
            nn.ConvTranspose2d(f*4, f*2, 3, 2, 1, output_padding=1, bias=False),
            nn.InstanceNorm2d(f*2), nn.ReLU(True),
            nn.ConvTranspose2d(f*2, f,   3, 2, 1, output_padding=1, bias=False),
            nn.InstanceNorm2d(f),   nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(f, out_ch, 7, 1, 0),
            nn.Tanh()
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


print('ResNet Generator defined.')

## Cell 7 — PatchGAN Discriminator

In [ ]:
class CycleDiscriminator(nn.Module):
    """
    PatchGAN discriminator for CycleGAN.
    Input: (B, 3, 128, 128)
    Output: (B, 1, H', W') patch scores — NO Sigmoid
    """
    def __init__(self, in_ch=3, f=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, f,   4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f,   f*2, 4, 2, 1, bias=False), nn.InstanceNorm2d(f*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f*2, f*4, 4, 2, 1, bias=False), nn.InstanceNorm2d(f*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f*4, f*8, 4, 1, 1, bias=False), nn.InstanceNorm2d(f*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f*8, 1,   4, 1, 1, bias=False)
            # No Sigmoid
        )
    def forward(self, x): return self.net(x)


def weights_init(m):
    cls = m.__class__.__name__
    if 'Conv' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)


# G_AB: Sketch → Photo
# G_BA: Photo  → Sketch
G_AB = ResNetGenerator().to(device)
G_BA = ResNetGenerator().to(device)
D_A  = CycleDiscriminator().to(device)  # classifies Sketch domain
D_B  = CycleDiscriminator().to(device)  # classifies Photo domain

if torch.cuda.device_count() > 1:
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A  = nn.DataParallel(D_A)
    D_B  = nn.DataParallel(D_B)
    print(f'DataParallel on {torch.cuda.device_count()} GPUs')

for m in [G_AB, G_BA, D_A, D_B]:
    m.apply(weights_init)

print('CycleGAN models initialised.')

## Cell 8 — Image Buffer (improves stability)

In [ ]:
class ImageBuffer:
    """
    Stores a history of 50 generated images.
    Discriminator trains on a mix of new and old images
    to reduce oscillation.
    """
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, images):
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(img)
                result.append(img)
            elif random.random() > 0.5:
                idx = random.randint(0, self.max_size-1)
                result.append(self.data[idx].clone())
                self.data[idx] = img
            else:
                result.append(img)
        return torch.cat(result).to(device)


fake_A_buffer = ImageBuffer()
fake_B_buffer = ImageBuffer()
print('Image buffers ready.')

## Cell 9 — Train CycleGAN

In [ ]:
# ── Optimizers ───────────────────────────────────────────
opt_G = optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()),
    lr=LR, betas=BETAS
)
opt_DA = optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_DB = optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

# LR decay after half of epochs
def lr_lambda(epoch):
    decay_start = EPOCHS // 2
    if epoch < decay_start:
        return 1.0
    return 1.0 - (epoch - decay_start) / (EPOCHS - decay_start + 1e-8)

sched_G  = optim.lr_scheduler.LambdaLR(opt_G,  lr_lambda)
sched_DA = optim.lr_scheduler.LambdaLR(opt_DA, lr_lambda)
sched_DB = optim.lr_scheduler.LambdaLR(opt_DB, lr_lambda)

# ── Losses ───────────────────────────────────────────────
adv_loss   = nn.MSELoss()    # LSGAN adversarial (more stable than BCE)
cycle_loss = nn.L1Loss()
id_loss    = nn.L1Loss()

G_losses, DA_losses, DB_losses, cycle_losses = [], [], [], []

# Fixed batch for visualization
fixed_A, fixed_B = next(iter(dataloader))
fixed_A = fixed_A[:2].to(device)
fixed_B = fixed_B[:2].to(device)

print('Starting CycleGAN Training...')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    ep_G, ep_DA, ep_DB, ep_cyc = 0.0, 0.0, 0.0, 0.0
    loop = tqdm(dataloader, desc=f'[CycleGAN] Epoch {epoch}/{EPOCHS}', leave=False)

    for real_A, real_B in loop:
        real_A = real_A.to(device)
        real_B = real_B.to(device)
        B      = real_A.size(0)

        real_label = torch.ones(B, 1, 14, 14, device=device)   # patch size
        fake_label = torch.zeros(B, 1, 14, 14, device=device)

        # ── Train Generators (G_AB + G_BA together) ───────
        opt_G.zero_grad()

        # Forward translations
        fake_B = G_AB(real_A)          # A → B
        fake_A = G_BA(real_B)          # B → A

        # Cycle reconstructions
        rec_A  = G_BA(fake_B)          # A → B → A
        rec_B  = G_AB(fake_A)          # B → A → B

        # Identity (optional but improves colour preservation)
        id_A   = G_BA(real_A)          # A → A
        id_B   = G_AB(real_B)          # B → B

        # Adversarial loss (fool discriminators)
        pred_fake_B = D_B(fake_B)
        pred_fake_A = D_A(fake_A)
        # Adjust label size to match discriminator output
        real_lbl = torch.ones_like(pred_fake_B)
        loss_adv_AB = adv_loss(pred_fake_B, real_lbl)
        loss_adv_BA = adv_loss(pred_fake_A, torch.ones_like(pred_fake_A))

        # Cycle consistency loss
        loss_cycle = (cycle_loss(rec_A, real_A) + cycle_loss(rec_B, real_B)) * LAMBDA_CYCLE

        # Identity loss
        loss_id = (id_loss(id_A, real_A) + id_loss(id_B, real_B)) * LAMBDA_ID

        loss_G = loss_adv_AB + loss_adv_BA + loss_cycle + loss_id
        loss_G.backward()
        opt_G.step()

        # ── Train Discriminator A ─────────────────────────
        opt_DA.zero_grad()
        fake_A_hist = fake_A_buffer.push_and_pop(fake_A.detach())
        pred_real   = D_A(real_A)
        pred_fake   = D_A(fake_A_hist)
        loss_DA = (adv_loss(pred_real, torch.ones_like(pred_real)) +
                   adv_loss(pred_fake, torch.zeros_like(pred_fake))) * 0.5
        loss_DA.backward()
        opt_DA.step()

        # ── Train Discriminator B ─────────────────────────
        opt_DB.zero_grad()
        fake_B_hist = fake_B_buffer.push_and_pop(fake_B.detach())
        pred_real   = D_B(real_B)
        pred_fake   = D_B(fake_B_hist)
        loss_DB = (adv_loss(pred_real, torch.ones_like(pred_real)) +
                   adv_loss(pred_fake, torch.zeros_like(pred_fake))) * 0.5
        loss_DB.backward()
        opt_DB.step()

        ep_G   += loss_G.item()
        ep_DA  += loss_DA.item()
        ep_DB  += loss_DB.item()
        ep_cyc += loss_cycle.item()
        loop.set_postfix(G=f'{loss_G.item():.3f}', DA=f'{loss_DA.item():.3f}',
                         DB=f'{loss_DB.item():.3f}', Cyc=f'{loss_cycle.item():.3f}')

    # LR schedule step
    sched_G.step(); sched_DA.step(); sched_DB.step()

    avg_G   = ep_G   / len(dataloader)
    avg_DA  = ep_DA  / len(dataloader)
    avg_DB  = ep_DB  / len(dataloader)
    avg_cyc = ep_cyc / len(dataloader)
    G_losses.append(avg_G); DA_losses.append(avg_DA)
    DB_losses.append(avg_DB); cycle_losses.append(avg_cyc)
    print(f'[CycleGAN] Epoch {epoch:03d}/{EPOCHS} | '
          f'G: {avg_G:.4f} | DA: {avg_DA:.4f} | DB: {avg_DB:.4f} | Cycle: {avg_cyc:.4f}')

    # Save sample translations
    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        fB = G_AB(fixed_A).cpu()
        fA = G_BA(fixed_B).cpu()
    G_AB.train(); G_BA.train()
    save_image(torch.cat([fixed_A.cpu(), fB], 0),
               f'outputs/cyclegan/AtoB/epoch_{epoch:03d}.png',
               normalize=True, value_range=(-1,1), nrow=2)
    save_image(torch.cat([fixed_B.cpu(), fA], 0),
               f'outputs/cyclegan/BtoA/epoch_{epoch:03d}.png',
               normalize=True, value_range=(-1,1), nrow=2)

    if epoch % SAVE_EVERY == 0:
        torch.save({'G_AB':G_AB.state_dict(),'G_BA':G_BA.state_dict(),
                    'D_A':D_A.state_dict(),'D_B':D_B.state_dict()},
                   f'checkpoints/cyclegan_{epoch:03d}.pth')

print('CycleGAN Training Complete!')

## Cell 10 — Loss Plots (3 curves)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(G_losses)+1)

axes[0].plot(ep, G_losses, color='#e74c3c', lw=2)
axes[0].set_title('Generator Loss (G_AB + G_BA)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(ep, DA_losses, label='D_A (Sketch)', color='#3498db', lw=2)
axes[1].plot(ep, DB_losses, label='D_B (Photo)',  color='#2ecc71', lw=2)
axes[1].set_title('Discriminator Losses', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, cycle_losses, color='#9b59b6', lw=2)
axes[2].set_title('Cycle Consistency Loss', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/cyclegan/training_losses.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Qualitative Results

In [ ]:
def show_cycle_results(n=5):
    G_AB.eval(); G_BA.eval()
    real_A_batch, real_B_batch = next(iter(dataloader))
    rA = real_A_batch[:n].to(device)
    rB = real_B_batch[:n].to(device)

    with torch.no_grad():
        fB   = G_AB(rA).cpu()           # Sketch → Photo
        rec  = G_BA(G_AB(rA)).cpu()     # Sketch → Photo → Sketch (reconstruction)
        fA   = G_BA(rB).cpu()           # Photo  → Sketch

    rA = rA.cpu(); rB = rB.cpu()

    fig, axes = plt.subplots(3, n, figsize=(n*3, 10))
    fig.suptitle('CycleGAN Results', fontsize=14, fontweight='bold')
    row_labels = ['Real Sketch (A)', 'Translated → Photo', 'Reconstructed ← Sketch']

    for col in range(n):
        for row, imgs in enumerate([rA, fB, rec]):
            img = imgs[col].permute(1,2,0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(row_labels[row], fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig('outputs/cyclegan/qualitative_AtoB.png', dpi=150, bbox_inches='tight')
    plt.show()
    G_AB.train(); G_BA.train()

show_cycle_results()

## Cell 12 — SSIM & PSNR

In [ ]:
!pip install -q scikit-image
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

def eval_cycle_metrics(n_batches=5):
    G_AB.eval(); G_BA.eval()
    ssim_scores, psnr_scores = [], []
    with torch.no_grad():
        for i, (rA, rB) in enumerate(dataloader):
            if i >= n_batches: break
            rA = rA.to(device)
            rec = G_BA(G_AB(rA)).cpu()
            rA  = rA.cpu()
            for j in range(rec.size(0)):
                r = ((rA[j].permute(1,2,0).numpy()  + 1) / 2 * 255).astype(np.uint8)
                f = ((rec[j].permute(1,2,0).numpy() + 1) / 2 * 255).astype(np.uint8)
                ssim_scores.append(ssim_fn(r, f, channel_axis=2, data_range=255))
                psnr_scores.append(psnr_fn(r, f, data_range=255))
    G_AB.train(); G_BA.train()
    return np.mean(ssim_scores), np.mean(psnr_scores)

ssim_val, psnr_val = eval_cycle_metrics()
print(f'Cycle Reconstruction SSIM : {ssim_val:.4f}')
print(f'Cycle Reconstruction PSNR : {psnr_val:.2f} dB')

## Cell 13 — Save Final Weights

In [ ]:
torch.save(G_AB.state_dict(), 'outputs/cyclegan_G_AB_final.pth')
torch.save(G_BA.state_dict(), 'outputs/cyclegan_G_BA_final.pth')

with open('outputs/cyclegan_loss_history.pkl', 'wb') as f:
    pickle.dump({
        'G': G_losses, 'DA': DA_losses,
        'DB': DB_losses, 'cycle': cycle_losses,
        'ssim': ssim_val, 'psnr': psnr_val
    }, f)

print('Saved:')
for f in sorted(os.listdir('outputs')): print(f'  {f}')